<a href="https://colab.research.google.com/github/DeepanshuSoni07/City-Energy-Consumption-Analysis-and-Prediction-System-Project/blob/main/Emotion_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
import nltk
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
stop_words = set(stopwords.words('english'))
important_words = {'not', 'no', 'never', 'neither', 'nor', 'none', 'but'}
stop_words = stop_words - important_words

In [4]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [w for w in text.split() if w not in stop_words]
    return " ".join(tokens)

In [5]:
print("Loading dataset...")
df = pd.read_csv("/content/tweet_emotions.csv")
df['clean_text'] = df['content'].apply(preprocess_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42
)

Loading dataset...


In [6]:
print("Training model...")
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
    ('clf', LogisticRegression(max_iter=1000))
])

Training model...


In [7]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [8]:
print("Model Training Complete!\n")
print(classification_report(y_test, y_pred))

Model Training Complete!

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00        19
     boredom       0.00      0.00      0.00        31
       empty       0.33      0.01      0.01       162
  enthusiasm       0.00      0.00      0.00       163
         fun       0.10      0.01      0.02       338
   happiness       0.35      0.39      0.37      1028
        hate       0.57      0.15      0.23       268
        love       0.51      0.40      0.45       762
     neutral       0.34      0.53      0.42      1740
      relief       0.32      0.02      0.03       352
     sadness       0.33      0.25      0.28      1046
    surprise       0.43      0.05      0.08       425
       worry       0.33      0.51      0.40      1666

    accuracy                           0.35      8000
   macro avg       0.28      0.18      0.18      8000
weighted avg       0.34      0.35      0.31      8000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
joblib.dump(pipeline, "emotion_model.pkl")

['emotion_model.pkl']

In [10]:
style = """
<style>
    .app-card {
        border: 1px solid #d1d1d1;
        border-radius: 12px;
        padding: 25px;
        background-color: #ffffff;
        box-shadow: 0px 4px 12px rgba(0,0,0,0.1);
    }
    .app-header {
        color: #202124;
        font-family: 'Helvetica Neue', Helvetica, Arial;
        font-size: 22px;
        font-weight: bold;
        text-align: center;
        margin-bottom: 20px;
    }
</style>
"""
display(HTML(style))


header = widgets.HTML('<div class="app-header">Text Emotion Analyzer</div>')

text_box = widgets.Textarea(
    value='',
    placeholder='',
    description='',
    layout=widgets.Layout(width="100%", height="120px")
)


button = widgets.Button(
    description="Analyze",
    button_style='info',
    layout=widgets.Layout(width="100%", height="40px", margin="15px 0px")
)


output_box = widgets.HTML(
    value='<div style="text-align: center; color: #888;">.</div>'
)


def on_click(b):
    user_text = text_box.value.strip()
    if not user_text:
        output_box.value = '<div style="color: #d93025; text-align: center;">Kripya kuch type karein.</div>'
        return

    # Prediction
    clean = preprocess_text(user_text)
    pred = pipeline.predict([clean])[0]

    # Result UI Update
    output_box.value = f"""
    <div style="background-color: #f1f3f4; padding: 15px; border-radius: 8px; border-top: 4px solid #1A73E8; text-align: center;">
        <p style="margin:0; font-size: 12px; color: #5f6368; letter-spacing: 1px;">PREDICTED CATEGORY</p>
        <h2 style="margin:5px 0; font-size: 24px; color: #202124;">{pred.upper()}</h2>
    </div>
    """

button.on_click(on_click)

# Layout assembly
app_layout = widgets.VBox([header, text_box, button, output_box],
                          layout=widgets.Layout(width='450px', margin='0 auto'))

# Final Container
final_container = widgets.Box([app_layout])
final_container.add_class("app-card")

display(final_container)

Box(children=(VBox(children=(HTML(value='<div class="app-header">Text Emotion Analyzer</div>'), Textarea(value…